# set up

In [1]:
from google.colab import drive
drive.mount('/content/drive')
dic = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/01OrginalData/')
data = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/02DataForAnalysis/')
result = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/03Results/')
brief = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/04DataForR/')

Mounted at /content/drive


In [2]:
import pandas as pd
import string
import re
import nltk

# social media groups:Health vs AD

Columns Overview:
- idx and id: Unique identifiers for each post.
- clean_text: The entire post's cleaned text.
- word: Words exploded from clean_text.

In [3]:
#read the data
ad_social = pd.read_csv( result + 'AD_social_lexicon.csv',engine='python')
ad_social

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,1.194189e+15,shocking new data has shown that people living...,shocking,['shocking'],1,['shocking'],['ADJ'],['JJ'],4.63,5.30,4.12,1.85
1,0,1.194189e+15,shocking new data has shown that people living...,new,['new'],1,['new'],['ADJ'],['JJ'],7.68,5.14,5.22,2.34
2,0,1.194189e+15,shocking new data has shown that people living...,data,['data'],1,['datum'],['NOUN'],['NNS'],NaN,NaN,NaN,1.70
3,0,1.194189e+15,shocking new data has shown that people living...,has,['has'],1,['have'],['VERB'],['VBZ'],NaN,NaN,NaN,2.34
4,0,1.194189e+15,shocking new data has shown that people living...,shown,['shown'],1,['show'],['VERB'],['VBN'],NaN,NaN,NaN,2.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
352467,2679,8.267263e+14,young people talk about their experiences of h...,having,['having'],1,['have'],['VERB'],['VBG'],NaN,NaN,NaN,2.26
352468,2679,8.267263e+14,young people talk about their experiences of h...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
352469,2679,8.267263e+14,young people talk about their experiences of h...,grandparent,['grandparent'],1,['grandparent'],['NOUN'],['NN'],NaN,NaN,NaN,0.92
352470,2679,8.267263e+14,young people talk about their experiences of h...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37


In [4]:
#read the data
health_social = pd.read_csv(result + 'Health_social_lexicon.csv',engine='python')
health_social

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,aarp,['aarp'],1,['aarp'],['NOUN'],['NN'],NaN,NaN,NaN,NaN
1,0,1.087023e+15,aarp volunteers gathered with user experience ...,volunteers,['volunteers'],1,['volunteer'],['NOUN'],['NNS'],NaN,NaN,NaN,1.66
2,0,1.087023e+15,aarp volunteers gathered with user experience ...,gathered,['gathered'],1,['gather'],['VERB'],['VBD'],NaN,NaN,NaN,1.98
3,0,1.087023e+15,aarp volunteers gathered with user experience ...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37
4,0,1.087023e+15,aarp volunteers gathered with user experience ...,user,['user'],1,['user'],['NOUN'],['NN'],3.67,3.21,5.56,1.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...
102623,3072,8.929571e+15,you may have heard that the federal government...,and,['and'],1,['and'],['CCONJ'],['CC'],NaN,NaN,NaN,2.37
102624,3072,8.929571e+15,you may have heard that the federal government...,how,['how'],1,['how'],['SCONJ'],['WRB'],NaN,NaN,NaN,2.18
102625,3072,8.929571e+15,you may have heard that the federal government...,to,['to'],1,['to'],['PART'],['TO'],NaN,NaN,NaN,2.37
102626,3072,8.929571e+15,you may have heard that the federal government...,get,['get'],1,['get'],['VERB'],['VB'],6.09,3.67,5.65,2.05


Since the word column in both DataFrames is exploded from the clean_text column, we can reduce the rows in ad_social to match the total row count of health_social.

In [5]:
# Reduce ad_social rows to match health_social's row count
ad_social_reduced = ad_social.iloc[:health_social.shape[0]].reset_index(drop=True)
ad_social_reduced
# ad_social.iloc[:health_social.shape[0]] selects only the first N rows of ad_social, where N equals the row count of health_social.

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,1.194189e+15,shocking new data has shown that people living...,shocking,['shocking'],1,['shocking'],['ADJ'],['JJ'],4.63,5.30,4.12,1.85
1,0,1.194189e+15,shocking new data has shown that people living...,new,['new'],1,['new'],['ADJ'],['JJ'],7.68,5.14,5.22,2.34
2,0,1.194189e+15,shocking new data has shown that people living...,data,['data'],1,['datum'],['NOUN'],['NNS'],NaN,NaN,NaN,1.70
3,0,1.194189e+15,shocking new data has shown that people living...,has,['has'],1,['have'],['VERB'],['VBZ'],NaN,NaN,NaN,2.34
4,0,1.194189e+15,shocking new data has shown that people living...,shown,['shown'],1,['show'],['VERB'],['VBN'],NaN,NaN,NaN,2.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
102623,472,8.213809e+14,‘we’d realised he was getting forgetful and st...,we’d,"['we', '’d']",2,"['we', '’d']","['PRON', 'VERB']","['PRP', 'VBD']",NaN,NaN,NaN,NaN
102624,472,8.213809e+14,‘we’d realised he was getting forgetful and st...,realised,['realised'],1,['realise'],['VERB'],['VBD'],NaN,NaN,NaN,1.75
102625,472,8.213809e+14,‘we’d realised he was getting forgetful and st...,he,['he'],1,['he'],['PRON'],['PRP'],NaN,NaN,NaN,2.20
102626,472,8.213809e+14,‘we’d realised he was getting forgetful and st...,was,['was'],1,['be'],['AUX'],['VBD'],NaN,NaN,NaN,2.33


In [6]:
ad_social_reduced['clean_text'][102627]

"‘we’d realised he was getting forgetful and starting to lose confidence. he had always been an outgoing, bubbly person, but he started not wanting to see people because he was embarrassed at forgetting them.'  samia egeh’s father, ali, 83, was diagnosed with vascular dementia about ten years ago after two strokes.   ‘at first, we all just carried on. we were quite ignorant ourselves and didn’t know much about dementia or how to deal with it, so we didn’t.’   following ali’s diagnosis, samia tried to improve her understanding of the condition to 'make sure my father had a better quality of life.’  however, samia has felt let down by some of the professional support that the family has been offered. in particular, a lack of culturally appropriate local services has left ali socially isolated.   ali grew up in somaliland before moving to cardiff as a teenager. he now prefers to communicate in the somali language, but there are no local services that meet his cultural needs, leaving him c

Above is the orginal text of the last post, netxt we need to verify whether the last post (idx=472 in the reduced ad_social) contains its full text. we can group by "id" and reconstruct the text for the last id.

In [7]:
#Find the `id` of the last post in the reduced ad_social DataFrame 获取最后一个post的id
last_id = ad_social_reduced.iloc[-1]['id']

#Filter rows corresponding to the last `id` 获取最后一个post对应的clean_text
last_post_rows = ad_social_reduced[ad_social_reduced['id'] == last_id]

In [8]:
# Reconstruct the full text by concatenating words
reconstructed_text = ' '.join(last_post_rows['word'])

print(f" (id={last_id}):\n{reconstructed_text}")

 (id=821380873308104.0):
we’d realised he was getting


In [9]:
# replace the content of the clean_text column for the last post (id equal to the last post) in the reduced ad_social
ad_social_reduced.loc[ad_social_reduced['id'] == last_id, 'clean_text'] = reconstructed_text

# Display the updated DataFrame for the last post
ad_social_reduced[ad_social_reduced['id'] == last_id]

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
102623,472,8.213809e+14,we’d realised he was getting,we’d,"['we', '’d']",2,"['we', '’d']","['PRON', 'VERB']","['PRP', 'VBD']",NaN,NaN,NaN,NaN
102624,472,8.213809e+14,we’d realised he was getting,realised,['realised'],1,['realise'],['VERB'],['VBD'],NaN,NaN,NaN,1.75
102625,472,8.213809e+14,we’d realised he was getting,he,['he'],1,['he'],['PRON'],['PRP'],NaN,NaN,NaN,2.20
102626,472,8.213809e+14,we’d realised he was getting,was,['was'],1,['be'],['AUX'],['VBD'],NaN,NaN,NaN,2.33
102627,472,8.213809e+14,we’d realised he was getting,getting,['getting'],1,['get'],['VERB'],['VBG'],NaN,NaN,NaN,1.97


In [21]:
#save the result
ad_social_reduced.to_csv(result + 'ad_social_lexicon_reduced.csv', index=False)

# Clinical diagnosis group: Health vs AD

In [11]:
#read the data
ad_clinical  = pd.read_csv(result + 'AD_clinical_lexicon.csv',engine='python')
ad_clinical

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm,['mhm'],1,['mhm'],['INTJ'],['UH'],NaN,NaN,NaN,NaN
1,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
2,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young,['young'],1,['young'],['ADJ'],['JJ'],6.31,4.09,5.60,2.00
3,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,boy,['boy'],1,['boy'],['PROPN'],['NNP'],5.84,4.11,5.50,1.61
4,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,going,['going'],1,['go'],['VERB'],['VBG'],NaN,NaN,NaN,2.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
62899,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,was,['was'],1,['be'],['AUX'],['VBD'],NaN,NaN,NaN,2.33
62900,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,caused,['caused'],1,['cause'],['VERB'],['VBD'],NaN,NaN,NaN,2.14
62901,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,by,['by'],1,['by'],['ADP'],['IN'],NaN,NaN,NaN,2.38
62902,194,Baycrest8961,mhm. yeah. um yes I can remember at least when...,long,['long'],1,['long'],['ADV'],['RB'],NaN,NaN,NaN,2.27


In [12]:
#read the data
health_clinical = pd.read_csv(result + 'Health_clinical_lexicon.csv',engine='python')
health_clinical

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
1,0,control-002-0,the scene is in the kitchen . the mother is wi...,scene,['scene'],1,['scene'],['NOUN'],['NN'],5.94,1.95,5.29,1.91
2,0,control-002-0,the scene is in the kitchen . the mother is wi...,is,['is'],1,['be'],['AUX'],['VBZ'],NaN,NaN,NaN,2.37
3,0,control-002-0,the scene is in the kitchen . the mother is wi...,in,['in'],1,['in'],['ADP'],['IN'],NaN,NaN,NaN,2.37
4,0,control-002-0,the scene is in the kitchen . the mother is wi...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7103,103,99-1,I see a woman drying dishes with her sink over...,dishes,['dishes'],1,['dish'],['NOUN'],['NNS'],NaN,NaN,NaN,1.26
7104,103,99-1,I see a woman drying dishes with her sink over...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37
7105,103,99-1,I see a woman drying dishes with her sink over...,her,['her'],1,['her'],['PRON'],['PRP$'],NaN,NaN,NaN,1.94
7106,103,99-1,I see a woman drying dishes with her sink over...,sink,['sink'],1,['sink'],['VERB'],['VB'],4.62,3.70,4.10,1.67


In [13]:
# Reduce ad_clinical rows to match health_clinical's row count
ad_clinical_reduced = ad_clinical.iloc[:health_clinical.shape[0]].reset_index(drop=True)
ad_clinical_reduced

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm,['mhm'],1,['mhm'],['INTJ'],['UH'],NaN,NaN,NaN,NaN
1,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
2,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young,['young'],1,['young'],['ADJ'],['JJ'],6.31,4.09,5.60,2.00
3,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,boy,['boy'],1,['boy'],['PROPN'],['NNP'],5.84,4.11,5.50,1.61
4,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,going,['going'],1,['go'],['VERB'],['VBG'],NaN,NaN,NaN,2.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7103,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
7104,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,other,['other'],1,['other'],['ADJ'],['JJ'],5.41,3.48,6.00,2.37
7105,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,cabinet,['cabinet'],1,['cabinet'],['NOUN'],['NN'],5.10,3.75,5.78,1.47
7106,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37


In [14]:
#Find the `id` of the last post in the reduced ad_clinical DataFrame
last_id = ad_clinical_reduced.iloc[-1]['id']

#Filter rows corresponding to the last `id`
last_post_rows = ad_clinical_reduced[ad_clinical_reduced['id'] == last_id]
last_post_rows

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
7090,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,cookie,['cookie'],1,['cookie'],['NOUN'],['NN'],7.32,4.70,6.10,1.53
7091,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,jar,['jar'],1,['jar'],['NOUN'],['NN'],5.71,2.77,5.26,1.55
7092,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
7093,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,boy,['boy'],1,['boy'],['PROPN'],['NNP'],5.84,4.11,5.50,1.61
7094,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,and,['and'],1,['and'],['CCONJ'],['CC'],NaN,NaN,NaN,2.37
7095,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
7096,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,girl,['girl'],1,['girl'],['NOUN'],['NN'],7.15,5.23,5.27,1.58
7097,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,and,['and'],1,['and'],['CCONJ'],['CC'],NaN,NaN,NaN,2.37
7098,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
7099,84,dementia-461-0,cookie jar . the boy and a girl and the ladder...,ladder,['ladder'],1,['ladder'],['NOUN'],['NN'],5.32,4.09,5.56,1.76


In [15]:
#最后一个post对应的orginal clean_text
original_text = last_post_rows['clean_text'].iloc[0]
original_text

"cookie jar . the boy and a girl and the ladder . the cabinet . and the other cabinet with the sink in it yes . the water coming out . the sink . the dishes on the cabinet . the lady drying dishes . the window . and you can see the trees outside the window . the curtains on the window . the water spilling over in the sink . I guess I mentioned the boy getting the cookies out of the jar . oh yeah the ladder's tipping over . the boy's on the ladder and the ladder's tipping over . and the girl's reaching to get a cookie . and the water's flowing down on the floor . you can see the grass out on the lawn ."

In [16]:
# Reconstruct the full text by concatenating words
reconstructed_text = ' '.join(last_post_rows['word'])

print(f" (id={last_id}):\n{reconstructed_text}")

 (id=dementia-461-0):
cookie jar the boy and a girl and the ladder the cabinet and the other cabinet with the


In [17]:
last_post_text = "cookie jar . the boy and a girl and the ladder . the cabinet . and the other cabinet with the "

In [18]:
# replace the content of the clean_text column for the last post (id equal to the last post) in the reduced ad_clinical
ad_clinical_reduced.loc[ad_clinical_reduced['id'] == last_id, 'clean_text'] = reconstructed_text

# Display the updated DataFrame for the last post
ad_clinical_reduced[ad_clinical_reduced['id'] == last_id]

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
7090,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,cookie,['cookie'],1,['cookie'],['NOUN'],['NN'],7.32,4.70,6.10,1.53
7091,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,jar,['jar'],1,['jar'],['NOUN'],['NN'],5.71,2.77,5.26,1.55
7092,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
7093,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,boy,['boy'],1,['boy'],['PROPN'],['NNP'],5.84,4.11,5.50,1.61
7094,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,and,['and'],1,['and'],['CCONJ'],['CC'],NaN,NaN,NaN,2.37
7095,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
7096,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,girl,['girl'],1,['girl'],['NOUN'],['NN'],7.15,5.23,5.27,1.58
7097,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,and,['and'],1,['and'],['CCONJ'],['CC'],NaN,NaN,NaN,2.37
7098,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
7099,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,ladder,['ladder'],1,['ladder'],['NOUN'],['NN'],5.32,4.09,5.56,1.76


In [20]:
#save the result
ad_clinical_reduced.to_csv(data + 'ad_clinical_lexicon_reduced.csv', index=False)

# final check

In [ ]:
print(f"Total columns in ad_social: {ad_social_reduced.shape[0]}")
print(f"Total columns in health_social: {health_social.shape[0]}")

Total columns in ad_social: 102628
Total columns in health_social: 102628


In [ ]:
print(f"Total columns in ad_clinical: {ad_clinical_reduced.shape[0]}")
print(f"Total columns in health_clinical: {health_clinical.shape[0]}")

Total columns in ad_clinical: 67930
Total columns in health_clinical: 67930
